# AtnLyze WAF — Transformer Training (Colab)

**Before running anything:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run Cell 1 (Mount Drive)
3. Upload `train_bert_augmented.csv` and `val_bert.csv` to Google Drive at: `MyDrive/atnlyze/data/`
4. Run remaining cells in order
5. After training, download `bert_waf.zip` from the Files panel (left sidebar)
6. Extract and copy the `bert_waf/` folder to your local `models/` directory

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DATA_DIR  = '/content/drive/MyDrive/atnlyze/data'
MODEL_DIR = '/content/bert_waf'
for f in ['train_bert_augmented.csv', 'val_bert.csv']:
    path = os.path.join(DATA_DIR, f)
    status = 'FOUND' if os.path.exists(path) else 'MISSING - upload to Drive first'
    print(f, ':', status)

In [ ]:
!pip install -q "transformers>=4.41,<5" datasets accelerate scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
import pandas as pd
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_bert_augmented.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val_bert.csv'))
print('Train:', len(train_df), '| benign:', (train_df['label']==0).sum(), '| malicious:', (train_df['label']==1).sum())
print('Val  :', len(val_df),   '| benign:', (val_df['label']==0).sum(),   '| malicious:', (val_df['label']==1).sum())
print('Sample malicious:')
print(train_df[train_df['label']==1]['structured_text'].iloc[0])

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

MODEL_NAME = 'distilbert-base-uncased'

class TransformerWAF:
    def __init__(self, model_dir=MODEL_DIR):
        self.model_dir = model_dir
        self.tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
        self.model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def _load_split(self, df):
        return Dataset.from_dict({'text': df['structured_text'].tolist(), 'label': df['label'].tolist()})

    def _tokenize(self, batch):
        return self.tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

    def train(self, train_df, val_df):
        train_ds = self._load_split(train_df).map(self._tokenize, batched=True)
        val_ds   = self._load_split(val_df).map(self._tokenize, batched=True)
        train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
        val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
        args = TrainingArguments(
            output_dir=self.model_dir,
            num_train_epochs=3,
            eval_strategy='epoch',
            save_strategy='epoch',
            per_device_train_batch_size=32,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            weight_decay=0.01,
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            save_total_limit=2,
            logging_strategy='steps',
            logging_steps=50,
            report_to='none',
            fp16=True,
        )
        trainer = Trainer(model=self.model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=self.compute_metrics)
        trainer.train()
        trainer.save_model(self.model_dir)
        self.tokenizer.save_pretrained(self.model_dir)
        print('Model saved to', self.model_dir)

    def compute_metrics(self, pred):
        labels = pred.label_ids
        probs  = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()
        preds  = (probs >= 0.5).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
        return {'accuracy': accuracy_score(labels, preds), 'precision': p, 'recall': r, 'f1': f1, 'roc_auc': roc_auc_score(labels, probs)}

print('TransformerWAF ready.')

In [ ]:
# Expected: 8-15 min on T4. Watch F1 — should reach 0.90+ by epoch 2.
waf = TransformerWAF(model_dir=MODEL_DIR)
waf.train(train_df, val_df)

In [ ]:
from transformers import pipeline
classifier = pipeline('text-classification', model=MODEL_DIR, tokenizer=MODEL_DIR, device=0)

test_cases = [
    ('[method] get [path] /dvwa/vulnerabilities/sqli/ [query] id=1 union select user,password from users-- [ua] mozilla/5.0 [referer] - [status] 302', 'MALICIOUS'),
    ('[method] get [path] /dvwa/vulnerabilities/xss_r/ [query] name=<script>alert(1)</script> [ua] mozilla/5.0 [referer] - [status] 302', 'MALICIOUS'),
    ('[method] get [path] /dvwa/vulnerabilities/fi/ [query] page=../../../etc/passwd [ua] mozilla/5.0 [referer] - [status] 302', 'MALICIOUS'),
    ('[method] get [path] /dvwa/vulnerabilities/exec/ [query] ip=127.0.0.1;cat /etc/passwd [ua] mozilla/5.0 [referer] - [status] 302', 'MALICIOUS'),
    ('[method] get [path] /dvwa/login.php [query] - [ua] mozilla/5.0 (windows nt 10.0) [referer] - [status] 200', 'BENIGN'),
    ('[method] get [path] /juice/rest/products/search [query] q=apple [ua] mozilla/5.0 [referer] - [status] 200', 'BENIGN'),
]

all_ok = True
print(f'{"Expected":<12} {"Got":<12} {"Score":<8} Input')
print('-'*90)
for text, expected in test_cases:
    r   = classifier(text, truncation=True, max_length=128)[0]
    got = 'MALICIOUS' if r['label'] == 'LABEL_1' else 'BENIGN'
    ok  = got == expected
    if not ok: all_ok = False
    print(f'{expected:<12} {got:<12} {r["score"]:<8.4f} {text[:55]}  [{"OK" if ok else "WRONG"}]')
print()
print('All correct!' if all_ok else 'Some wrong - post results before downloading.')

In [ ]:
import shutil
shutil.make_archive('/content/bert_waf', 'zip', '/content', 'bert_waf')
size_mb = os.path.getsize('/content/bert_waf.zip') / 1e6
print('bert_waf.zip ready:', round(size_mb, 1), 'MB')
print('Download: Files panel -> bert_waf.zip -> right-click -> Download')
print('Then extract and copy bert_waf/ into your local models/ directory.')

In [ ]:
import shutil, os
drive_path = '/content/drive/MyDrive/atnlyze/models'
os.makedirs(drive_path, exist_ok=True)
shutil.copy('/content/bert_waf.zip', os.path.join(drive_path, 'bert_waf.zip'))
print('Backup saved to Google Drive:', drive_path)
print('Safe to close Colab now.')